# Week 7 — Basic Biostatistics in Python  
## Summary statistics, inferential statistics, and interpreting results

**Big idea:** Biostatistics helps us move from “what do these data look like?” to “what does this pattern probably mean?”  

**What you will practice**
- Understanding the difference between **summary** and **inferential** statistics
- Using common Python packages for basic biostatistics
- Calculating means, medians, proportions, standard deviations, and confidence intervals
- Running a few very common statistical tests
- Interpreting p-values, confidence intervals, and effect sizes
- Applying this to a simple **self-efficacy survey** example

**Packages we will use**
- `pandas` → tables and grouped summaries
- `numpy` → numerical calculations
- `matplotlib` → simple plots
- `scipy.stats` → common statistical tests
- `statsmodels` → confidence intervals and regression-style summaries

**For today’s mindset**
You do **not** need to memorize everything.  
You mainly need to learn:
1. what kinds of questions statistics answers,
2. which package helps with which task,
3. how to read the output without panicking.


## 0) Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

np.random.seed(42)

## 1) Summary stats vs inferential stats

### Summary statistics
These **describe the data you already have**.

Examples:
- mean age of a cohort
- median hospital length of stay
- proportion of patients with hypertension
- standard deviation of systolic blood pressure

### Inferential statistics
These help you make a claim **beyond just the sample in front of you**.

Examples:
- is blood pressure different between two groups?
- is smoking associated with disease status?
- is a change before vs after an intervention likely to be real?
- how uncertain is our estimate?

### Easy way to remember it
- **Summary stats = “What do my data look like?”**
- **Inferential stats = “What might this mean in the larger world?”**


## 2) Make a toy clinical dataset

We will make a small fake dataset so we can practice the workflow safely.

Imagine:
- two groups: `control` and `program`
- outcome: systolic blood pressure
- a few other basic variables


In [ ]:
n = 80

clinical = pd.DataFrame({
    "group": np.random.choice(["control", "program"], size=n),
    "age": np.random.normal(56, 12, size=n).round(0),
    "female": np.random.choice([0, 1], size=n, p=[0.45, 0.55]),
})

# Create blood pressure with a small average improvement in the program group
clinical["sbp"] = (
    np.random.normal(132, 12, size=n)
    - (clinical["group"] == "program") * 6
    + (clinical["age"] - 56) * 0.15
).round(1)

clinical["diabetes"] = np.random.choice([0, 1], size=n, p=[0.72, 0.28])

clinical.head()

## 3) First look at the data

Before doing statistics, always inspect:
- shape
- column names
- data types
- a quick preview


In [ ]:
print("shape:", clinical.shape)
print("\ncolumns:", list(clinical.columns))
print("\ndtypes:")
print(clinical.dtypes)

## 4) Basic summary statistics with `pandas`

For numeric columns, `describe()` gives a fast overview:
- count
- mean
- standard deviation
- min / max
- quartiles


In [ ]:
clinical.describe(numeric_only=True)

### Grouped summaries

A lot of biomedical analysis is really just:
1. split into groups
2. summarize each group
3. compare


In [ ]:
clinical.groupby("group")["sbp"].agg(["count", "mean", "median", "std", "min", "max"])

### Categorical summaries

For binary or categorical variables, we often want:
- counts
- proportions


In [ ]:
count_table = clinical["group"].value_counts()
prop_table = clinical["group"].value_counts(normalize=True)

print("Counts:")
print(count_table)

print("\nProportions:")
print(prop_table.round(3))

### Cross-tabulation

This is useful when comparing counts across categories.


In [ ]:
pd.crosstab(clinical["group"], clinical["diabetes"], margins=True)

## 5) Visualizing the data

Plots are not separate from statistics.  
They help you see the distribution before you run a test.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for group_name, subset in clinical.groupby("group"):
    ax.hist(subset["sbp"], bins=10, alpha=0.6, label=group_name)

ax.set_title("Distribution of systolic blood pressure by group")
ax.set_xlabel("Systolic blood pressure")
ax.set_ylabel("Count")
ax.legend()
plt.show()

## 6) A very common inferential question:
### “Are these two group means different?”

A classic first test is the **independent samples t-test**.

Use it when:
- you have **two groups**
- the outcome is **continuous**
- you want to compare group means

We will compare `sbp` between `control` and `program`.


In [ ]:
control_sbp = clinical.loc[clinical["group"] == "control", "sbp"]
program_sbp = clinical.loc[clinical["group"] == "program", "sbp"]

t_stat, p_value = stats.ttest_ind(control_sbp, program_sbp, equal_var=False)

print("t statistic:", round(t_stat, 3))
print("p-value:", round(p_value, 4))

### Interpreting the p-value

A p-value is **not**:
- the probability your hypothesis is true
- the probability the result happened “by chance”

A practical beginner interpretation:

> **If there were really no difference between groups, how surprising would data this extreme be?**

Common classroom shorthand:
- `p < 0.05` → often called “statistically significant”
- `p >= 0.05` → not enough evidence to say there is a difference

But statistical significance is **not the same thing** as clinical importance.


## 7) Confidence intervals

Confidence intervals tell us the **range of plausible values** for an estimate.

We can calculate a 95% confidence interval for mean systolic blood pressure in each group.


In [ ]:
def mean_ci(series, alpha=0.05):
    series = pd.Series(series).dropna()
    mean = series.mean()
    sem = stats.sem(series)
    low, high = stats.t.interval(
        confidence=1 - alpha,
        df=len(series) - 1,
        loc=mean,
        scale=sem
    )
    return pd.Series({
        "mean": mean,
        "ci_low": low,
        "ci_high": high
    })

clinical.groupby("group")["sbp"].apply(mean_ci).unstack().round(2)

### Why confidence intervals matter

They give more information than a p-value alone:
- the **estimated size** of the effect
- the **uncertainty** around that estimate

That is often much closer to how clinicians actually think.


## 8) Categorical inference with the chi-square test

Now let’s ask:

**Is diabetes status distributed differently across the two groups?**

This is a count-based question, so a **chi-square test** is appropriate.


In [ ]:
table = pd.crosstab(clinical["group"], clinical["diabetes"])
chi2, p, dof, expected = stats.chi2_contingency(table)

print("Contingency table:")
display(table)

print("chi-square:", round(chi2, 3))
print("degrees of freedom:", dof)
print("p-value:", round(p, 4))

## 9) Correlation

Sometimes we want to know whether two continuous variables tend to move together.

Example:
- age and systolic blood pressure

We will use **Pearson correlation** for a simple linear relationship.


In [ ]:
r, p = stats.pearsonr(clinical["age"], clinical["sbp"])

print("Pearson r:", round(r, 3))
print("p-value:", round(p, 4))

### Interpreting correlation
- `r` near `0` → weak linear relationship
- `r` near `1` → strong positive linear relationship
- `r` near `-1` → strong negative linear relationship

But correlation does **not** prove causation.


## 10) A beginner-friendly regression example with `statsmodels`

Regression is powerful because it lets us estimate a relationship while accounting for other variables.

Here we will model systolic blood pressure as a function of:
- group
- age
- diabetes


In [ ]:
model = smf.ols("sbp ~ C(group) + age + diabetes", data=clinical).fit()
model.summary()

### How to read this output without getting overwhelmed

Focus on a few pieces first:
- **coef** → estimated effect size
- **P>|t|** → p-value for that term
- **[0.025, 0.975]** → 95% confidence interval

For `C(group)[T.program]`, a negative coefficient suggests the program group has lower systolic blood pressure than the control group, after adjusting for the other variables in the model.


## 11) Interpreting results: a simple checklist

Whenever you run a test, ask these in order:

1. **What exactly was compared?**  
   Example: mean systolic blood pressure in control vs program.

2. **What was the effect size or estimated difference?**  
   Do not stop at “significant” or “not significant.”

3. **What is the uncertainty?**  
   Look at the confidence interval.

4. **What is the p-value?**  
   This tells you how compatible the data are with the null model.

5. **Does the result matter clinically or educationally?**  
   Statistical significance is only one part of interpretation.

6. **Does the test match the data type and study design?**  
   Continuous vs categorical, paired vs unpaired, small vs large sample.


## 13) Which package should I reach for?

### `pandas`
Use for:
- loading data
- cleaning data
- grouped summaries
- quick tables

### `numpy`
Use for:
- arrays
- numerical operations
- simulation
- mathematical functions

### `matplotlib`
Use for:
- histograms
- scatterplots
- bar charts
- quick visual checks

### `scipy.stats`
Use for:
- t-tests
- chi-square tests
- correlations
- basic confidence-interval building blocks

### `statsmodels`
Use for:
- regression
- richer statistical summaries
- confidence intervals
- more formal modeling


## 15) Mini exercises

Try these on your own:

1. Compare **age** between the two groups with an independent t-test.  
2. Make a bar chart of the proportion with diabetes in each group.  
3. Compute a 95% confidence interval for age in the full sample.  
4. Change the toy survey data and see how the paired t-test changes.  
5. In one sentence each, explain:
   - what a p-value is
   - what a confidence interval is
   - why “statistically significant” does not automatically mean “important”
